# Customer Response Prediction - Production-Ready System
## Advanced ML Pipeline with Explainability, Segmentation & Deployment

**Objective**: Build a production-grade customer response prediction model with:
- Advanced feature engineering
- SMOTE for imbalance handling
- Multiple model comparison
- SHAP explainability
- Customer segmentation
- Model persistence
- Streamlit deployment-ready code

## 1. Import Libraries & Setup

In [ ]:
# Core libraries
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
plt.style.use('seaborn-v0_8-darkgrid')

# ML preprocessing
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder

# Imbalance handling
from imblearn.over_sampling import SMOTE

# Models
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from xgboost import XGBClassifier

# Evaluation metrics
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, classification_report, confusion_matrix,
    roc_curve
)

# Clustering
from sklearn.cluster import KMeans

# Explainability
import shap

# Model persistence
import pickle
from datetime import datetime

print("✓ All libraries imported successfully")

## 2. Data Loading & Initial Exploration

In [ ]:
# Load dataset
df = pd.read_csv('marketing_campaign.csv', sep='\t')

print(f"Dataset shape: {df.shape}")
print(f"\nTarget distribution:\n{df['Response'].value_counts()}")
print(f"\nImbalance ratio: {df['Response'].value_counts()[0] / df['Response'].value_counts()[1]:.2f}:1")

df.head()

In [ ]:
# Check data quality
print("Missing values:\n", df.isnull().sum()[df.isnull().sum() > 0])
print("\nData types:\n", df.dtypes.value_counts())

## 3. Data Preprocessing (Modular Function)

**Key improvements:**
- Proper handling of missing values
- Date feature extraction
- Outlier detection and capping
- Categorical encoding

In [ ]:
def preprocess_data(df, is_training=True):
    """
    Comprehensive data preprocessing pipeline
    
    Args:
        df: Input dataframe
        is_training: Whether this is training data (affects target extraction)
    
    Returns:
        Processed dataframe
    """
    df = df.copy()
    
    # 1. Remove ID column (not predictive)
    if 'ID' in df.columns:
        df.drop('ID', axis=1, inplace=True)
    
    # 2. Extract target variable if training
    if is_training and 'Response' in df.columns:
        target = df['Response']
        df.drop('Response', axis=1, inplace=True)
    else:
        target = None
    
    # 3. Handle missing values
    # Income has some missing values - fill with median
    df['Income'] = df['Income'].fillna(df['Income'].median())
    
    # 4. Date processing - extract customer tenure
    if 'Dt_Customer' in df.columns:
        df['Dt_Customer'] = pd.to_datetime(df['Dt_Customer'], dayfirst=True)
        reference_date = df['Dt_Customer'].max()
        df['Customer_Days'] = (reference_date - df['Dt_Customer']).dt.days
        df.drop('Dt_Customer', axis=1, inplace=True)
    
    # 5. Remove constant columns (Z_CostContact, Z_Revenue)
    constant_cols = [col for col in df.columns if df[col].nunique() == 1]
    if constant_cols:
        df.drop(constant_cols, axis=1, inplace=True)
    
    # 6. Handle outliers in Income (cap at 99th percentile)
    income_99 = df['Income'].quantile(0.99)
    df['Income'] = df['Income'].clip(upper=income_99)
    
    # 7. Handle Year_Birth outliers (some unrealistic values)
    df = df[df['Year_Birth'] > 1900]  # Remove anomalies
    df['Age'] = 2014 - df['Year_Birth']  # Approximate year from dataset
    df.drop('Year_Birth', axis=1, inplace=True)
    
    # 8. Encode categorical variables
    # Education: ordinal encoding
    education_map = {'Basic': 1, '2n Cycle': 2, 'Graduation': 3, 'Master': 4, 'PhD': 5}
    df['Education'] = df['Education'].map(education_map)
    
    # Marital Status: binary encoding (partnered vs single)
    df['Is_Partnered'] = df['Marital_Status'].isin(['Married', 'Together']).astype(int)
    df.drop('Marital_Status', axis=1, inplace=True)
    
    print(f"✓ Preprocessing complete. Shape: {df.shape}")
    
    return df, target

# Apply preprocessing
df_processed, y = preprocess_data(df, is_training=True)
df_processed.head()

## 4. Advanced Feature Engineering

**Business-driven features that improve model performance:**

1. **Customer Lifetime Value (CLV)**: Total spending across all categories
   - *Why it helps*: High-value customers are more likely to respond to campaigns

2. **Engagement Score**: Combination of purchase frequency and channel usage
   - *Why it helps*: Engaged customers show higher conversion rates

3. **Purchase Frequency**: Total number of purchases across all channels
   - *Why it helps*: Frequent buyers are more responsive to marketing

4. **Spending per Purchase**: Average transaction value
   - *Why it helps*: Indicates customer quality and willingness to spend

5. **Campaign Acceptance Rate**: Historical campaign response
   - *Why it helps*: Past behavior predicts future response

6. **Family Burden**: Total dependents (kids + teens)
   - *Why it helps*: Affects disposable income and purchase decisions

In [ ]:
def engineer_features(df):
    """
    Create advanced behavioral and business features
    
    Args:
        df: Preprocessed dataframe
    
    Returns:
        Dataframe with engineered features
    """
    df = df.copy()
    
    # 1. Customer Lifetime Value (CLV) - Total spending
    spending_cols = ['MntWines', 'MntFruits', 'MntMeatProducts', 
                     'MntFishProducts', 'MntSweetProducts', 'MntGoldProds']
    df['CLV'] = df[spending_cols].sum(axis=1)
    
    # 2. Purchase Frequency - Total purchases
    purchase_cols = ['NumDealsPurchases', 'NumWebPurchases', 
                     'NumCatalogPurchases', 'NumStorePurchases']
    df['Purchase_Frequency'] = df[purchase_cols].sum(axis=1)
    
    # 3. Average Spending per Purchase
    df['Avg_Spending_Per_Purchase'] = df['CLV'] / (df['Purchase_Frequency'] + 1)  # +1 to avoid division by zero
    
    # 4. Engagement Score (composite metric)
    # Normalizes purchases and web visits into 0-100 score
    purchase_score = (df['Purchase_Frequency'] / df['Purchase_Frequency'].max()) * 50
    web_visit_score = ((10 - df['NumWebVisitsMonth']) / 10) * 50  # Inverse: fewer visits = more engaged
    df['Engagement_Score'] = purchase_score + web_visit_score
    
    # 5. Campaign Acceptance Rate
    campaign_cols = ['AcceptedCmp1', 'AcceptedCmp2', 'AcceptedCmp3', 
                     'AcceptedCmp4', 'AcceptedCmp5']
    df['Campaign_Acceptance_Rate'] = df[campaign_cols].sum(axis=1) / len(campaign_cols)
    
    # 6. Family Burden
    df['Total_Dependents'] = df['Kidhome'] + df['Teenhome']
    
    # 7. Income per Dependent (purchasing power)
    df['Income_Per_Member'] = df['Income'] / (df['Total_Dependents'] + df['Is_Partnered'] + 1)
    
    # 8. Recency Risk (inverse of Recency - high recency = low engagement)
    df['Recency_Risk'] = df['Recency'] / df['Recency'].max()
    
    # 9. Channel Preference Diversity (uses multiple channels = more engaged)
    channel_cols = ['NumWebPurchases', 'NumCatalogPurchases', 'NumStorePurchases']
    df['Channel_Diversity'] = (df[channel_cols] > 0).sum(axis=1)
    
    # 10. Product Category Diversity
    df['Product_Diversity'] = (df[spending_cols] > 0).sum(axis=1)
    
    print("✓ Feature engineering complete")
    print(f"  - New features created: {len(df.columns) - len(df_processed.columns)}")
    print(f"  - Total features: {len(df.columns)}")
    
    return df

# Apply feature engineering
df_features = engineer_features(df_processed)

# Display new features
new_features = ['CLV', 'Purchase_Frequency', 'Avg_Spending_Per_Purchase', 
                'Engagement_Score', 'Campaign_Acceptance_Rate', 'Income_Per_Member']
df_features[new_features].describe()

## 5. Data Splitting & SMOTE for Imbalance Handling

**Why SMOTE is better than random resampling:**
- **Random Oversampling**: Duplicates minority class samples → model memorizes duplicates
- **SMOTE**: Creates synthetic samples by interpolating between minority class neighbors
  - More diverse training data
  - Better generalization
  - Reduces overfitting
  - Preserves feature relationships

In [ ]:
# Prepare features and target
X = df_features

# Train-test split (80-20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Before SMOTE:")
print(f"  Training set: {X_train.shape}")
print(f"  Class distribution: {y_train.value_counts().to_dict()}")
print(f"  Imbalance ratio: {y_train.value_counts()[0]/y_train.value_counts()[1]:.2f}:1")

# Apply SMOTE to training data only
smote = SMOTE(random_state=42, k_neighbors=5)
X_train_balanced, y_train_balanced = smote.fit_resample(X_train, y_train)

print("\nAfter SMOTE:")
print(f"  Training set: {X_train_balanced.shape}")
print(f"  Class distribution: {pd.Series(y_train_balanced).value_counts().to_dict()}")
print(f"  Imbalance ratio: 1:1 (balanced)")
print(f"\n✓ SMOTE applied: {X_train_balanced.shape[0] - X_train.shape[0]} synthetic samples created")

## 6. Feature Scaling

In [ ]:
# Standardize features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_balanced)
X_test_scaled = scaler.transform(X_test)

print("✓ Features scaled using StandardScaler")

## 7. Model Training & Comparison

**Models evaluated:**
1. **Logistic Regression**: Baseline linear model
2. **Random Forest**: Ensemble of decision trees
3. **XGBoost**: Gradient boosting (industry standard)

In [ ]:
def train_models(X_train, y_train, X_test, y_test):
    """
    Train and evaluate multiple classification models
    
    Returns:
        Dictionary of trained models and performance metrics
    """
    models = {}
    results = []
    
    # 1. Logistic Regression
    print("Training Logistic Regression...")
    lr = LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced')
    lr.fit(X_train, y_train)
    models['Logistic Regression'] = lr
    
    # 2. Random Forest
    print("Training Random Forest...")
    rf = RandomForestClassifier(
        n_estimators=200,
        max_depth=10,
        min_samples_split=5,
        random_state=42,
        n_jobs=-1
    )
    rf.fit(X_train, y_train)
    models['Random Forest'] = rf
    
    # 3. XGBoost
    print("Training XGBoost...")
    xgb = XGBClassifier(
        n_estimators=300,
        learning_rate=0.1,
        max_depth=5,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        eval_metric='logloss'
    )
    xgb.fit(X_train, y_train)
    models['XGBoost'] = xgb
    
    # Evaluate all models
    for name, model in models.items():
        y_pred = model.predict(X_test)
        y_proba = model.predict_proba(X_test)[:, 1]
        
        results.append({
            'Model': name,
            'Accuracy': accuracy_score(y_test, y_pred),
            'Precision': precision_score(y_test, y_pred),
            'Recall': recall_score(y_test, y_pred),
            'F1-Score': f1_score(y_test, y_pred),
            'ROC-AUC': roc_auc_score(y_test, y_proba)
        })
    
    print("\n✓ All models trained successfully")
    return models, pd.DataFrame(results)

# Train models
trained_models, performance_df = train_models(
    X_train_scaled, y_train_balanced,
    X_test_scaled, y_test
)

## 8. Model Performance Comparison

In [ ]:
# Display comparison table
performance_df_styled = performance_df.round(4).sort_values('ROC-AUC', ascending=False)
print("\n" + "="*80)
print("MODEL PERFORMANCE COMPARISON")
print("="*80)
print(performance_df_styled.to_string(index=False))
print("="*80)

# Find best model
best_model_name = performance_df_styled.iloc[0]['Model']
print(f"\n🏆 Best Model: {best_model_name} (ROC-AUC: {performance_df_styled.iloc[0]['ROC-AUC']:.4f})")

In [ ]:
# Visualize comparison
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Plot 1: All metrics comparison
metrics = ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC']
performance_df.set_index('Model')[metrics].plot(kind='bar', ax=axes[0], rot=0)
axes[0].set_title('Model Performance Comparison', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Score')
axes[0].set_ylim([0, 1])
axes[0].legend(loc='lower right')
axes[0].grid(axis='y', alpha=0.3)

# Plot 2: ROC-AUC focus
performance_df.plot(x='Model', y='ROC-AUC', kind='barh', ax=axes[1], color='#2ecc71', legend=False)
axes[1].set_title('ROC-AUC Score Comparison', fontsize=14, fontweight='bold')
axes[1].set_xlabel('ROC-AUC Score')
axes[1].set_ylabel('')
axes[1].set_xlim([0.8, 1.0])
axes[1].grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.show()

## 9. Detailed Classification Report (Best Model)

In [ ]:
# Get best model
best_model = trained_models[best_model_name]

# Predictions
y_pred = best_model.predict(X_test_scaled)
y_proba = best_model.predict_proba(X_test_scaled)[:, 1]

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Confusion Matrix
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0], 
            xticklabels=['Not Respond', 'Respond'],
            yticklabels=['Not Respond', 'Respond'])
axes[0].set_title(f'Confusion Matrix - {best_model_name}', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Actual')
axes[0].set_xlabel('Predicted')

# ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_proba)
axes[1].plot(fpr, tpr, linewidth=2, label=f'ROC Curve (AUC = {roc_auc_score(y_test, y_proba):.3f})')
axes[1].plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random Classifier')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC Curve', fontsize=14, fontweight='bold')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

# Print classification report
print("\n" + "="*60)
print(f"CLASSIFICATION REPORT - {best_model_name}")
print("="*60)
print(classification_report(y_test, y_pred, target_names=['Not Respond', 'Respond']))

## 10. Feature Importance Analysis

In [ ]:
# Get feature importance
if hasattr(best_model, 'feature_importances_'):
    importance_df = pd.DataFrame({
        'Feature': X.columns,
        'Importance': best_model.feature_importances_
    }).sort_values('Importance', ascending=False)
    
    # Plot top 15 features
    plt.figure(figsize=(12, 6))
    sns.barplot(data=importance_df.head(15), x='Importance', y='Feature', palette='viridis')
    plt.title(f'Top 15 Most Important Features - {best_model_name}', fontsize=14, fontweight='bold')
    plt.xlabel('Feature Importance')
    plt.tight_layout()
    plt.show()
    
    print("\nTop 10 Most Important Features:")
    print(importance_df.head(10).to_string(index=False))
else:
    print(f"{best_model_name} does not provide feature importance scores")

## 11. SHAP Explainability

**SHAP (SHapley Additive exPlanations)** provides:
- Individual prediction explanations
- Feature contribution to each prediction
- Global feature importance
- Better interpretability for stakeholders

In [ ]:
# Initialize SHAP explainer
print("Calculating SHAP values (this may take a minute)...")

# Use TreeExplainer for tree-based models, LinearExplainer for linear models
if best_model_name in ['Random Forest', 'XGBoost']:
    explainer = shap.TreeExplainer(best_model)
    shap_values = explainer.shap_values(X_test_scaled[:100])  # Use subset for speed
    
    # For binary classification, SHAP returns values for class 1
    if isinstance(shap_values, list):
        shap_values = shap_values[1]
else:
    explainer = shap.LinearExplainer(best_model, X_train_scaled)
    shap_values = explainer.shap_values(X_test_scaled[:100])

print("✓ SHAP values calculated")

In [ ]:
# SHAP Summary Plot (global feature importance)
plt.figure(figsize=(12, 8))
shap.summary_plot(shap_values, X_test_scaled[:100], feature_names=X.columns, show=False)
plt.title('SHAP Feature Importance - Global View', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### SHAP Explanation for Individual Prediction

In [ ]:
# Explain a single prediction (first test sample)
sample_idx = 0
sample_prediction = best_model.predict_proba(X_test_scaled[sample_idx:sample_idx+1])[0]

print(f"Sample Prediction:")
print(f"  - Probability of Response: {sample_prediction[1]:.2%}")
print(f"  - Prediction: {'Will Respond' if sample_prediction[1] > 0.5 else 'Will Not Respond'}")

# Force plot for single prediction
shap.initjs()
shap.force_plot(
    explainer.expected_value if not isinstance(explainer.expected_value, np.ndarray) else explainer.expected_value[1],
    shap_values[sample_idx],
    X_test_scaled[sample_idx],
    feature_names=X.columns
)

## 12. Customer Segmentation using KMeans

**Business value**: Group customers into segments for targeted marketing strategies

In [ ]:
def perform_customer_segmentation(df, n_clusters=4):
    """
    Segment customers using KMeans clustering
    
    Args:
        df: Feature dataframe
        n_clusters: Number of customer segments
    
    Returns:
        Dataframe with cluster assignments
    """
    # Select key features for clustering
    clustering_features = ['CLV', 'Purchase_Frequency', 'Engagement_Score', 
                          'Income', 'Campaign_Acceptance_Rate']
    
    X_cluster = df[clustering_features].copy()
    
    # Scale features
    scaler_cluster = StandardScaler()
    X_cluster_scaled = scaler_cluster.fit_transform(X_cluster)
    
    # Apply KMeans
    kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
    clusters = kmeans.fit_predict(X_cluster_scaled)
    
    # Add cluster labels
    df['Segment'] = clusters
    
    return df, kmeans, clustering_features

# Apply segmentation
df_segmented, kmeans_model, cluster_features = perform_customer_segmentation(df_features, n_clusters=4)

print(f"✓ Customer segmentation complete")
print(f"\nSegment distribution:")
print(df_segmented['Segment'].value_counts().sort_index())

### Segment Profiling

In [ ]:
# Create segment profiles
segment_profiles = df_segmented.groupby('Segment')[cluster_features].mean()

print("\n" + "="*80)
print("CUSTOMER SEGMENT PROFILES")
print("="*80)
print(segment_profiles.round(2))
print("="*80)

In [ ]:
# Visualize segments
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for idx, feature in enumerate(cluster_features):
    df_segmented.groupby('Segment')[feature].mean().plot(kind='bar', ax=axes[idx], color='steelblue')
    axes[idx].set_title(f'{feature} by Segment', fontweight='bold')
    axes[idx].set_xlabel('Segment')
    axes[idx].set_ylabel('Average Value')
    axes[idx].grid(axis='y', alpha=0.3)

# Response rate by segment
response_by_segment = df_segmented.groupby('Segment')[y.name].mean()
response_by_segment.plot(kind='bar', ax=axes[5], color='coral')
axes[5].set_title('Response Rate by Segment', fontweight='bold')
axes[5].set_xlabel('Segment')
axes[5].set_ylabel('Response Rate')
axes[5].set_ylim([0, 1])
axes[5].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

### Business Interpretation of Segments

In [ ]:
# Define segment names based on profiles
def interpret_segments(df_segmented, segment_profiles):
    """
    Interpret customer segments in business terms
    """
    interpretations = []
    
    for seg in range(len(segment_profiles)):
        profile = segment_profiles.loc[seg]
        
        # Segment characteristics
        clv = profile['CLV']
        freq = profile['Purchase_Frequency']
        engagement = profile['Engagement_Score']
        income = profile['Income']
        acceptance = profile['Campaign_Acceptance_Rate']
        
        # Response rate
        response_rate = df_segmented[df_segmented['Segment'] == seg][y.name].mean()
        
        # Determine segment type
        if clv > segment_profiles['CLV'].median() and engagement > segment_profiles['Engagement_Score'].median():
            segment_name = "Premium Champions"
            description = "High-value, highly engaged customers"
            strategy = "VIP treatment, exclusive offers, loyalty rewards"
        elif freq > segment_profiles['Purchase_Frequency'].median() and acceptance > 0.3:
            segment_name = "Engaged Buyers"
            description = "Frequent purchasers, responsive to campaigns"
            strategy = "Cross-sell, upsell, personalized recommendations"
        elif income > segment_profiles['Income'].median() and clv < segment_profiles['CLV'].median():
            segment_name = "Potential Stars"
            description = "High income but low spending - untapped potential"
            strategy = "Targeted campaigns, incentivize first purchase, onboarding"
        else:
            segment_name = "At-Risk / Low-Value"
            description = "Low engagement, low spending"
            strategy = "Re-engagement campaigns, discounts, win-back offers"
        
        interpretations.append({
            'Segment': seg,
            'Name': segment_name,
            'Description': description,
            'CLV': f"${clv:,.0f}",
            'Purchase_Freq': f"{freq:.1f}",
            'Response_Rate': f"{response_rate:.1%}",
            'Strategy': strategy
        })
    
    return pd.DataFrame(interpretations)

segment_interpretation = interpret_segments(df_segmented, segment_profiles)

print("\n" + "="*100)
print("BUSINESS INTERPRETATION OF CUSTOMER SEGMENTS")
print("="*100)
for _, row in segment_interpretation.iterrows():
    print(f"\n📊 Segment {row['Segment']}: {row['Name']}")
    print(f"   Description: {row['Description']}")
    print(f"   Avg CLV: {row['CLV']} | Avg Purchases: {row['Purchase_Freq']} | Response Rate: {row['Response_Rate']}")
    print(f"   📈 Strategy: {row['Strategy']}")
print("="*100)

## 13. Enhanced Prediction Function with Recommendations

In [ ]:
def predict_with_recommendation(input_data, model, scaler, feature_columns):
    """
    Make prediction with business recommendations
    
    Args:
        input_data: Dictionary of customer features
        model: Trained model
        scaler: Fitted scaler
        feature_columns: List of feature names in correct order
    
    Returns:
        Dictionary with prediction, probability, risk level, and recommendation
    """
    # Convert to dataframe
    input_df = pd.DataFrame([input_data])
    
    # Ensure all features are present
    for col in feature_columns:
        if col not in input_df.columns:
            input_df[col] = 0  # Default value for missing features
    
    # Reorder columns to match training data
    input_df = input_df[feature_columns]
    
    # Scale features
    input_scaled = scaler.transform(input_df)
    
    # Make prediction
    probability = model.predict_proba(input_scaled)[0][1]
    prediction = "Will Respond" if probability > 0.5 else "Will Not Respond"
    
    # Determine risk level and recommendation
    if probability >= 0.75:
        risk_level = "High Probability"
        recommendation = "✅ Prime Target: Include in premium campaigns with personalized offers"
        action = "Target"
    elif probability >= 0.5:
        risk_level = "Medium Probability"
        recommendation = "⚠️ Potential Responder: Include in standard campaigns with incentives"
        action = "Engage"
    elif probability >= 0.25:
        risk_level = "Low Probability"
        recommendation = "🔄 At-Risk: Re-engagement campaign with discount offers"
        action = "Re-engage"
    else:
        risk_level = "Very Low Probability"
        recommendation = "❌ Low Priority: Exclude from expensive campaigns, minimal outreach"
        action = "Deprioritize"
    
    return {
        'prediction': prediction,
        'probability': round(probability * 100, 2),
        'risk_level': risk_level,
        'recommendation': recommendation,
        'action': action
    }

print("✓ Enhanced prediction function ready")

### Test Prediction Function

In [ ]:
# Test with sample customer
sample_customer = {
    'Income': 75000,
    'Recency': 15,
    'CLV': 1200,
    'Purchase_Frequency': 18,
    'Engagement_Score': 65,
    'Campaign_Acceptance_Rate': 0.4,
    'Age': 45,
    'Customer_Days': 800
}

result = predict_with_recommendation(sample_customer, best_model, scaler, X.columns)

print("\n" + "="*80)
print("SAMPLE PREDICTION RESULT")
print("="*80)
print(f"Prediction: {result['prediction']}")
print(f"Probability: {result['probability']}%")
print(f"Risk Level: {result['risk_level']}")
print(f"Recommended Action: {result['action']}")
print(f"\n{result['recommendation']}")
print("="*80)

## 14. Model Persistence (Save & Load)

In [ ]:
# Save model artifacts
def save_model_artifacts(model, scaler, feature_columns, model_name="best_model"):
    """
    Save trained model and preprocessing artifacts
    """
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    
    artifacts = {
        'model': model,
        'scaler': scaler,
        'feature_columns': feature_columns,
        'model_name': model_name,
        'timestamp': timestamp
    }
    
    filename = f"model_artifacts_{timestamp}.pkl"
    
    with open(filename, 'wb') as f:
        pickle.dump(artifacts, f)
    
    print(f"✓ Model artifacts saved to: {filename}")
    return filename

# Save the model
model_file = save_model_artifacts(best_model, scaler, X.columns.tolist(), best_model_name)

In [ ]:
# Load model artifacts
def load_model_artifacts(filename):
    """
    Load saved model and preprocessing artifacts
    """
    with open(filename, 'rb') as f:
        artifacts = pickle.load(f)
    
    print(f"✓ Model loaded: {artifacts['model_name']}")
    print(f"  Saved at: {artifacts['timestamp']}")
    
    return artifacts

# Test loading
loaded_artifacts = load_model_artifacts(model_file)
loaded_model = loaded_artifacts['model']
loaded_scaler = loaded_artifacts['scaler']
loaded_features = loaded_artifacts['feature_columns']

## 15. Production-Ready Modular Code

Below is the complete modular code structure for production deployment:

In [ ]:
# This cell contains all modular functions in one place for easy export

class CustomerResponsePredictor:
    """
    Production-ready customer response prediction system
    """
    
    def __init__(self):
        self.model = None
        self.scaler = None
        self.feature_columns = None
        self.model_name = None
    
    def preprocess(self, df, is_training=True):
        """Data preprocessing pipeline"""
        return preprocess_data(df, is_training)
    
    def engineer_features(self, df):
        """Feature engineering pipeline"""
        return engineer_features(df)
    
    def train(self, X_train, y_train, X_test, y_test):
        """Train multiple models and select best"""
        models, performance = train_models(X_train, y_train, X_test, y_test)
        best_model_name = performance.sort_values('ROC-AUC', ascending=False).iloc[0]['Model']
        self.model = models[best_model_name]
        self.model_name = best_model_name
        return performance
    
    def predict(self, input_data):
        """Make prediction with recommendations"""
        if self.model is None:
            raise ValueError("Model not trained. Call train() first.")
        return predict_with_recommendation(input_data, self.model, self.scaler, self.feature_columns)
    
    def save(self, filename):
        """Save model artifacts"""
        return save_model_artifacts(self.model, self.scaler, self.feature_columns, self.model_name)
    
    def load(self, filename):
        """Load model artifacts"""
        artifacts = load_model_artifacts(filename)
        self.model = artifacts['model']
        self.scaler = artifacts['scaler']
        self.feature_columns = artifacts['feature_columns']
        self.model_name = artifacts['model_name']

print("✓ CustomerResponsePredictor class defined")

## 16. Summary & Key Insights

### Improvements Made:

1. **Advanced Feature Engineering**: Created 10+ behavioral features (CLV, Engagement Score, etc.)
2. **SMOTE for Imbalance**: Replaced random resampling with synthetic minority oversampling
3. **Model Comparison**: Evaluated 3 models with comprehensive metrics
4. **SHAP Explainability**: Added global and local feature importance analysis
5. **Customer Segmentation**: KMeans clustering with business interpretations
6. **Enhanced Predictions**: Probability + Risk Level + Actionable Recommendations
7. **Model Persistence**: Save/load functionality for deployment
8. **Modular Code**: Production-ready class structure

### Next Steps:
- Deploy with Streamlit UI (see separate file)
- Integrate with CRM systems
- A/B test campaign strategies by segment
- Monitor model performance and retrain periodically